# 06. Feedback Text Analysis

This notebook summarizes tester ratings, categories, pages, and recurring words. It is deliberately simple and interpretable.

In [1]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

WORKSPACE = Path.cwd()
if WORKSPACE.name == "notebooks":
    WORKSPACE = WORKSPACE.parent

sys.path.insert(0, str(WORKSPACE / "src"))

from venture_ds import (
    load_exports,
    load_optional_export,
    safe_rate,
    wilson_interval,
    ordered_session_funnel,
    add_sample_warning,
)

exports = load_exports(WORKSPACE / "data" / "raw")
events = exports["events"]
sessions = exports["sessions"]
feedback = exports["feedback"]
inventory = exports["inventory"]

exports["_files"]

,dataset,path,found
0,events,C:\Users\ppmal\Documents\Projects\venture-prod...,True
1,sessions,C:\Users\ppmal\Documents\Projects\venture-prod...,True
2,feedback,C:\Users\ppmal\Documents\Projects\venture-prod...,True
3,inventory,C:\Users\ppmal\Documents\Projects\venture-prod...,True


In [2]:
if feedback.empty:
    print("No feedback export found.")
else:
    display(
        feedback.groupby("category")
        .agg(
            responses=("feedback_id", "size"),
            average_rating=("rating", "mean"),
        )
        .sort_values("responses", ascending=False)
    )
    display(
        feedback.groupby("page_path")
        .agg(
            responses=("feedback_id", "size"),
            average_rating=("rating", "mean"),
        )
        .sort_values(["average_rating", "responses"])
        .head(20)
    )

No feedback export found.


In [3]:
import re
from collections import Counter

STOP_WORDS = {
    "the", "and", "that", "this", "with", "for", "was", "are",
    "but", "not", "you", "your", "have", "from", "too", "very",
    "just", "can", "could", "would", "into", "when", "what",
}

if not feedback.empty:
    words = []
    for message in feedback["message"].dropna().astype(str):
        words.extend(
            token for token in re.findall(r"[a-z']{3,}", message.lower())
            if token not in STOP_WORDS
        )

    keyword_table = pd.DataFrame(
        Counter(words).most_common(30),
        columns=["word", "count"],
    )
    display(keyword_table)

## Privacy note

Remove or mask any contact information before presenting text examples. Quotes should be paraphrased unless the tester has clearly consented to public attribution.